# Imports

In [1]:
import os
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
import torch.nn as nn
from torch.utils.data import random_split
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm

# Loader

In [2]:
CLASSES = ["drinking", "packaging", "resting", "lifting", "cutting", "using-computer-mouse"] #removed "mixing" "walking"

class IMUDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.samples = self._prepare_dataset()

    def _prepare_dataset(self):
        samples = []
        for dir_name in os.listdir(self.root_dir):
            label = dir_name.split('_')[0]
            full_path = os.path.join(self.root_dir, dir_name)
            if os.path.isdir(full_path):
                samples.append((full_path, label))
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label_str = self.samples[idx]
        imu_data = pd.read_csv(os.path.join(path, 'out.csv'))
        
        #process puck data
        imu_data.drop(columns=['temp'])

        # Step 2: Split into leg and hand data
        #leg = imu_data[imu_data['sender'] == 'leg'].drop(columns=['sender']).sort_values('timestamp')
        imu_data = imu_data[imu_data['sender'] == 'hand'].drop(columns=['sender']).sort_values('timestamp')
        
        # Step 3: Rename columns to differentiate
        #leg = leg.add_prefix('leg_')
        #hand = hand.add_prefix('hand_')
        
        # Step 4: Rename timestamp columns back to match for merging
        #leg = leg.rename(columns={'leg_timestamp': 'timestamp'})
        #hand = hand.rename(columns={'hand_timestamp': 'timestamp'})
        
        # Step 5: Merge on timestamp
        #imu_data = pd.merge_asof(hand, leg, on='timestamp', direction='nearest', tolerance=2)
        
        # Step 6: Drop unnecessary columns
        #imu_data = imu_data.drop(columns=['timestamp', 'timestamp', 'leg_temp', 'hand_temp'])
        imu_data = imu_data.drop(columns=['timestamp', 'temp'])
        
        # Step 7: Rename columns to match IWMSHA
        #imu_data = imu_data.rename(columns={'leg_gyro_x':'gx2', 'leg_gyro_y':'gy2', 'leg_gyro_z':'gz2', 'leg_accel_x':'ax2', 'leg_accel_y':'ay2', 'leg_accel_z':'az2',   'hand_gyro_x':'gx1',  'hand_gyro_y':'gy1',  'hand_gyro_z':'gz1', 'hand_accel_x':'ax1',  'hand_accel_y':'ay1',  'hand_accel_z':'az1' })
        imu_data = imu_data.rename(columns={ 'gyro_x':'gx1',  'gyro_y':'gy1',  'gyro_z':'gz1', 'accel_x':'ax1',  'accel_y':'ay1',  'accel_z':'az1' })
        
        
        #imu_data = imu_data[['ax1','ay1','az1','gx1','gy1','gz1','ax2','ay2','az2','gx2','gy2','gz2']]
        #imu_data = imu_data[['ax1','ay1','az1','gx1','gy1','gz1']]
        imu_data = imu_data.loc[:, ~imu_data.columns.str.contains('^Unnamed')]
        
        #print(imu_data.head())
        
        # Convert label string to numeric
        label = CLASSES.index(label_str)
        
        # takes each parse as one entry for the label and hence less data to be split into multiple :) 
        # Convert IMU data and label to tensors
        
        imu_tensor = torch.tensor(imu_data.values, dtype=torch.float32)
        label_tensor = torch.tensor(label, dtype=torch.long)

        sample = {
            'imu_data': imu_tensor,
            'label': label_tensor
        }
    
        return sample
    

# Model

In [3]:
class IMUOnlyClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(IMUOnlyClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.dropout = nn.Dropout(p=0.1)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x shape: (batch_size, sequence_length, input_size)
        out, _ = self.lstm(x)
        out = self.dropout(out)
        out = self.fc(out[:, -1, :])  # Get the last time step
        return out

# Split

In [4]:
dataset = IMUDataset(root_dir="../puck_data_collect/")

# Assume 'dataset' is your initialized PyTorch dataset
dataset_size = len(dataset)
train_size = int(0.7 * dataset_size)
test_size = dataset_size - train_size
generator1 = torch.Generator().manual_seed(42)
# Split the dataset into train and test sets
train_dataset, test_dataset = random_split(dataset,[train_size,test_size],generator1)

# You can now use these subsets with a DataLoader
train_loader = torch.utils.data.DataLoader(train_dataset, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, shuffle=False)



# Train

In [5]:
input_size = 6 # For example, 2*(3 axes for gyro and 3 for accel)
hidden_size = 35  # Example size, can be tuned
num_layers = 2  # Number of LSTM layers
num_classes = len(CLASSES)
torch.manual_seed(42)
model = IMUOnlyClassifier(input_size, hidden_size, num_layers, num_classes)

In [6]:
# Assuming model is already defined and initialized
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=0.001)

num_epochs = 60  # Define the number of epochs
for epoch in tqdm(range(num_epochs)):
    model.train()  # Set the model to training mode
    for i, data in enumerate(train_loader):
        # Forward pass
        outputs = model(data['imu_data'])
        loss = criterion(outputs, data['label'])
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (i+1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')
            

  0%|          | 0/60 [00:00<?, ?it/s]

Epoch [1/60], Step [100/133], Loss: 1.6671


  2%|▏         | 1/60 [00:02<02:06,  2.15s/it]

Epoch [2/60], Step [100/133], Loss: 1.8089


  3%|▎         | 2/60 [00:04<02:01,  2.09s/it]

Epoch [3/60], Step [100/133], Loss: 1.7074


... (111 outputs left out)

Epoch [59/60], Step [100/133], Loss: 0.0875


 98%|█████████▊| 59/60 [02:02<00:02,  2.09s/it]

Epoch [60/60], Step [100/133], Loss: 0.0947


100%|██████████| 60/60 [02:04<00:00,  2.08s/it]


# Eval

In [7]:
from sklearn.metrics import accuracy_score, recall_score, f1_score


# Function to convert tensors to numpy arrays
def to_numpy(tensor):
    return tensor.cpu().detach().numpy()

# Evaluation
model.eval()
all_labels = []
all_predictions = []

with torch.no_grad():
    for i, data in enumerate(test_loader):
        # Forward pass
        outputs = model(data['imu_data'])
        _, predicted = torch.max(outputs, 1)
        
        all_labels.extend(to_numpy(data['label']))
        all_predictions.extend(to_numpy(predicted))

# Convert all labels and predictions to numpy arrays for sklearn functions
all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)

# Calculate and print metrics
print(classification_report(all_labels, all_predictions, zero_division=0))

# Calculate and print the confusion matrix
conf_matrix = confusion_matrix(all_labels, all_predictions)
print("Confusion Matrix:")
print(conf_matrix)

accuracy = accuracy_score(all_labels, all_predictions)
print("Accuracy:", accuracy)

recall = recall_score(all_labels,all_predictions, average='weighted', zero_division=0)
print("recall:", recall)
f1score = f1_score(all_labels,all_predictions, average='weighted')
print("F1_score:", f1score)

              precision    recall  f1-score   support

           0       1.00      0.67      0.80        12
           1       1.00      0.89      0.94         9
           2       0.67      0.75      0.71         8
           3       0.82      1.00      0.90         9
           4       0.90      0.82      0.86        11
           5       0.73      1.00      0.84         8

    accuracy                           0.84        57
   macro avg       0.85      0.85      0.84        57
weighted avg       0.87      0.84      0.84        57

Confusion Matrix:
[[8 0 2 0 1 1]
 [0 8 1 0 0 0]
 [0 0 6 2 0 0]
 [0 0 0 9 0 0]
 [0 0 0 0 9 2]
 [0 0 0 0 0 8]]
Accuracy: 0.8421052631578947
recall: 0.8421052631578947
F1_score: 0.8418080805729494


# Save

In [8]:
# Save the entire model
torch.save(model, '../model/imu_only_rnn_imwsha_100.pth')

# Load

# To load the model later
loaded_model = torch.load('../model/imu_only_rnn.pth')